# Phase 9 --- Walk-Forward Backtest Engine

**Objective:** Execute a rigorous walk-forward backtest of the Black-Litterman and Mean-CVaR
dynamic factor allocation strategies against static benchmarks. All signals use information
available at time $t$ only; returns are realised at $t+1$.

**Protocol:**
- **Warm-up:** Months 1--60 (2005-01 to 2009-12) -- estimation only, no live trading
- **Live:** Months 61--252 (2010-01 to 2025-12) -- out-of-sample evaluation
- **Transaction cost:** TC = 25 bps one-way (`TC_FACTOR = 0.0025`)

**Inputs:** `factor_returns_full.parquet`, `bl_weights_timeseries.parquet`,
`cvar_weights_timeseries.parquet`, `regime_probabilities.parquet`

**Outputs:** `backtest_returns.parquet`, `backtest_nav.parquet`,
`outputs/tables/backtest_performance.csv`, figures (NAV, drawdown, rolling Sharpe, etc.)

---
## 9.1 Setup & Load

In [1]:
import sys
import os
import warnings
import logging
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.stats import norm

# Project imports
sys.path.insert(0, os.path.abspath('..'))
from src.config import (
    PROCESSED_DIR, FIGURES_DIR, TABLES_DIR,
    FACTOR_RETURNS_FULL_FILE, BL_WEIGHTS_FILE, CVAR_WEIGHTS_FILE,
    REGIME_PROBS_FILE, FACTOR_RETURNS_FILE,
    BACKTEST_RETURNS_FILE, BACKTEST_NAV_FILE, BACKTEST_PERF_FILE,
    TC_FACTOR, RANDOM_STATE, COLORS, FACTOR_LIVE_START
)
from src.validation import validate_parquet, quick_data_check, check_nan_propagation
from src.risk_metrics import compute_all_metrics, benchmark_relative_metrics
from src.visualization import (
    setup_style, save_fig, plot_cumulative_returns,
    plot_drawdown, plot_weight_evolution
)

# Style and logging
setup_style()
warnings.filterwarnings('ignore', category=FutureWarning)
np.random.seed(RANDOM_STATE)

logging.basicConfig(
    filename=f'../logs/phase_09_{datetime.now():%Y%m%d_%H%M}.log',
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s'
)
logger = logging.getLogger(__name__)
logger.info('Phase 9 --- Walk-Forward Backtest Engine started')

print(f'TC_FACTOR = {TC_FACTOR} ({TC_FACTOR * 10_000:.0f} bps one-way)')
print(f'RANDOM_STATE = {RANDOM_STATE}')
print(f'FACTOR_LIVE_START = {FACTOR_LIVE_START}')

TC_FACTOR = 0.0025 (25 bps one-way)
RANDOM_STATE = 42
FACTOR_LIVE_START = 2010-01-01


In [2]:
# ---- Load factor returns (monthly, decimal) ----
factor_returns_full = pd.read_parquet(FACTOR_RETURNS_FULL_FILE)
validate_parquet(
    factor_returns_full,
    expected_cols=['hml', 'umd', 'rmw', 'lowvol'],
    min_rows=200,
    date_index=True,
    label='factor_returns_full'
)
quick_data_check(factor_returns_full, 'factor_returns_full')

# Core 4 factors used in allocation
FACTORS = ['hml', 'umd', 'rmw', 'lowvol']
factor_ret = factor_returns_full[FACTORS].copy()

# ---- Load Fama-French data for market benchmark ----
ff_data = pd.read_parquet(FACTOR_RETURNS_FILE)
validate_parquet(ff_data, expected_cols=['mkt_rf', 'rf'], min_rows=200,
                 date_index=True, label='factor_returns')
quick_data_check(ff_data, 'factor_returns (FF data)')

=== factor_returns_full ===
  Shape: (232, 4)
  Index: DatetimeIndex, range: 2005-01-31 00:00:00 → 2024-04-30 00:00:00
  Duplicates: 0
  NaN total: 0 (0.0%)
  Inf total: 0
  Dtypes: {dtype('float64'): np.int64(4)}
  Sorted: True

=== factor_returns (FF data) ===
  Shape: (232, 7)
  Index: DatetimeIndex, range: 2005-01-31 00:00:00 → 2024-04-30 00:00:00
  Duplicates: 0
  NaN total: 0 (0.0%)
  Inf total: 0
  Dtypes: {dtype('float64'): np.int64(7)}
  Sorted: True



In [3]:
# ---- Load dynamic weights from Phase 7 & 8 ----
bl_weights = pd.read_parquet(BL_WEIGHTS_FILE)
validate_parquet(bl_weights, min_rows=100, date_index=True, label='bl_weights')
quick_data_check(bl_weights, 'bl_weights_timeseries')

cvar_weights = pd.read_parquet(CVAR_WEIGHTS_FILE)
validate_parquet(cvar_weights, min_rows=100, date_index=True, label='cvar_weights')
quick_data_check(cvar_weights, 'cvar_weights_timeseries')

# ---- Load regime probabilities from Phase 4 ----
regime_probs = pd.read_parquet(REGIME_PROBS_FILE)
validate_parquet(regime_probs, min_rows=150, date_index=True, label='regime_probs')
quick_data_check(regime_probs, 'regime_probabilities')

=== bl_weights_timeseries ===
  Shape: (125, 4)
  Index: DatetimeIndex, range: 2013-11-30 00:00:00 → 2024-03-31 00:00:00
  Duplicates: 0
  NaN total: 0 (0.0%)
  Inf total: 0
  Dtypes: {dtype('float64'): np.int64(4)}
  Sorted: True

=== cvar_weights_timeseries ===
  Shape: (185, 4)
  Index: DatetimeIndex, range: 2008-11-30 00:00:00 → 2024-03-31 00:00:00
  Duplicates: 0
  NaN total: 0 (0.0%)
  Inf total: 0
  Dtypes: {dtype('float64'): np.int64(4)}
  Sorted: True

=== regime_probabilities ===
  Shape: (186, 4)
  Index: DatetimeIndex, range: 2008-11-30 00:00:00 → 2024-04-30 00:00:00
  Duplicates: 0
  NaN total: 0 (0.0%)
  Inf total: 0
  Dtypes: {dtype('float64'): np.int64(3), dtype('O'): np.int64(1)}
  Sorted: True



In [4]:
# ---- Align all datasets to common monthly date index ----
# Normalise to month-end
factor_ret.index = factor_ret.index + pd.offsets.MonthEnd(0)
ff_data.index = ff_data.index + pd.offsets.MonthEnd(0)
bl_weights.index = bl_weights.index + pd.offsets.MonthEnd(0)
cvar_weights.index = cvar_weights.index + pd.offsets.MonthEnd(0)
regime_probs.index = regime_probs.index + pd.offsets.MonthEnd(0)

# Inner join across all datasets
common_idx = (
    factor_ret.index
    .intersection(ff_data.index)
    .intersection(bl_weights.index)
    .intersection(cvar_weights.index)
    .intersection(regime_probs.index)
).sort_values()

# Remove duplicates
common_idx = common_idx[~common_idx.duplicated()]

factor_ret = factor_ret.loc[common_idx]
ff_data = ff_data.loc[common_idx]
bl_weights = bl_weights.loc[common_idx]
cvar_weights = cvar_weights.loc[common_idx]
regime_probs = regime_probs.loc[common_idx]

print(f'Aligned date range: {common_idx.min()} to {common_idx.max()}')
print(f'Total months: {len(common_idx)}')
print(f'Factors: {FACTORS}')

# Verify no NaN in returns
check_nan_propagation(factor_ret, 'aligned factor returns')
assert factor_ret.index.is_unique, 'Duplicate dates in factor returns!'

Aligned date range: 2013-11-30 00:00:00 to 2024-03-31 00:00:00
Total months: 125
Factors: ['hml', 'umd', 'rmw', 'lowvol']


---
## 9.2 Walk-Forward Backtest Protocol

For each month $t$ from 60 to $T-1$:
1. **ESTIMATE**: expanding window $[1, t]$ (already done in Phases 4--8)
2. **SIGNAL**: BL/CVaR weights at time $t$ (pre-computed, no look-ahead)
3. **EXECUTE**: $r_{\text{net},t+1} = \sum_i w_{i,t} \times r_{i,t+1} - \text{TC} \times \sum_i |w_{i,t} - w_{i,t-1}|$

NAV: $V_{t+1} = V_t \times (1 + r_{\text{net},t+1})$

In [5]:
def run_walk_forward_backtest(
    factor_returns: pd.DataFrame,
    weights_ts: pd.DataFrame,
    tc: float = TC_FACTOR,
    warmup_months: int = 60,
    strategy_name: str = 'Strategy'
) -> dict:
    """
    Walk-forward backtest engine.

    Parameters
    ----------
    factor_returns : pd.DataFrame
        Monthly factor returns (T x N), decimal.
    weights_ts : pd.DataFrame
        Weight time series (T x N). Weights at t used for return at t+1.
    tc : float
        One-way transaction cost (0.0025 = 25 bps).
    warmup_months : int
        Number of months reserved for warm-up (no live P&L).
    strategy_name : str
        Name for logging.

    Returns
    -------
    dict with keys: 'returns', 'nav', 'turnover', 'tc_cost', 'weights'
    """
    dates = factor_returns.index
    n_factors = factor_returns.shape[1]
    factors = factor_returns.columns.tolist()

    # Align weight columns to factor columns
    # Handle different column naming conventions
    w_cols = weights_ts.columns.tolist()
    if all(c.startswith('w_') for c in w_cols):
        col_map = {f'w_{f}': f for f in factors}
        weights_ts = weights_ts.rename(columns=col_map)
    weights_ts = weights_ts[factors]

    # Storage
    net_returns = []
    turnover_series = []
    tc_cost_series = []
    weight_history = []

    # Initialise: equal-weight before first trade
    prev_weights = np.ones(n_factors) / n_factors

    for t in range(warmup_months, len(dates) - 1):
        date_t = dates[t]
        date_t1 = dates[t + 1]

        # Signal: weights determined at time t
        if date_t in weights_ts.index:
            w_t = weights_ts.loc[date_t].values.astype(float)
        else:
            w_t = prev_weights  # fallback

        # Ensure valid weights
        w_t = np.maximum(w_t, 0.0)
        if w_t.sum() > 0:
            w_t = w_t / w_t.sum()
        else:
            w_t = np.ones(n_factors) / n_factors

        # Turnover and transaction cost
        turnover_t = np.abs(w_t - prev_weights).sum()
        tc_cost_t = tc * turnover_t

        # Execute: gross return at t+1 using weights at t
        r_t1 = factor_returns.loc[date_t1].values
        gross_return = np.dot(w_t, r_t1)
        net_return = gross_return - tc_cost_t

        net_returns.append({'date': date_t1, 'return': net_return})
        turnover_series.append({'date': date_t1, 'turnover': turnover_t})
        tc_cost_series.append({'date': date_t1, 'tc_cost': tc_cost_t})
        weight_history.append({'date': date_t, **dict(zip(factors, w_t))})

        # Update prev_weights for next iteration
        prev_weights = w_t.copy()

    # Assemble results
    ret_df = pd.DataFrame(net_returns).set_index('date')['return']
    ret_df.name = strategy_name

    nav = 100.0 * (1 + ret_df).cumprod()
    nav.name = strategy_name

    turnover_df = pd.DataFrame(turnover_series).set_index('date')['turnover']
    tc_df = pd.DataFrame(tc_cost_series).set_index('date')['tc_cost']
    weight_df = pd.DataFrame(weight_history).set_index('date')

    # Summary stats
    n_months = len(ret_df)
    n_years = n_months / 12
    ann_turnover = turnover_df.sum() / n_years
    ann_tc_drag = tc_df.sum() / n_years

    print(f'  {strategy_name}: {n_months} live months, '
          f'Ann. Turnover={ann_turnover:.1%}, Ann. TC drag={ann_tc_drag*1e4:.1f} bps')

    return {
        'returns': ret_df,
        'nav': nav,
        'turnover': turnover_df,
        'tc_cost': tc_df,
        'weights': weight_df,
        'ann_turnover': ann_turnover,
        'ann_tc_drag': ann_tc_drag
    }


print('Walk-forward engine defined.')

Walk-forward engine defined.


In [6]:
# ---- Determine warm-up cutoff ----
# Warm-up: first 60 months. Live starts at month 61.
WARMUP_MONTHS = 60
live_start_idx = WARMUP_MONTHS
live_start_date = factor_ret.index[live_start_idx]

print(f'Total months in sample: {len(factor_ret)}')
print(f'Warm-up: months 1-{WARMUP_MONTHS} ({factor_ret.index[0].strftime("%Y-%m")} '
      f'to {factor_ret.index[WARMUP_MONTHS-1].strftime("%Y-%m")})')
print(f'Live starts at: {live_start_date.strftime("%Y-%m")} (month {WARMUP_MONTHS+1})')
print(f'Live months: {len(factor_ret) - WARMUP_MONTHS - 1}')

Total months in sample: 125
Warm-up: months 1-60 (2013-11 to 2018-10)
Live starts at: 2018-11 (month 61)
Live months: 64


In [7]:
# ---- Run backtest for dynamic strategies ----
print('Running walk-forward backtests...')
print('=' * 60)

bl_result = run_walk_forward_backtest(
    factor_ret, bl_weights, tc=TC_FACTOR,
    warmup_months=WARMUP_MONTHS, strategy_name='BL Dynamic'
)

cvar_result = run_walk_forward_backtest(
    factor_ret, cvar_weights, tc=TC_FACTOR,
    warmup_months=WARMUP_MONTHS, strategy_name='CVaR Dynamic'
)

print('=' * 60)
print('Dynamic strategy backtests complete.')

Running walk-forward backtests...
  BL Dynamic: 64 live months, Ann. Turnover=0.0%, Ann. TC drag=0.0 bps
  CVaR Dynamic: 64 live months, Ann. Turnover=14.4%, Ann. TC drag=3.6 bps
Dynamic strategy backtests complete.


---
## 9.3 Benchmarks

1. **Equal-Weight Static:** $w = [0.25]^4$ rebalanced monthly
2. **Inverse-Vol Static:** risk parity without regime tilts
3. **Market (Mkt-RF + RF):** S&P 500 total return
4. **60/40:** 60% (Mkt-RF + RF) + 40% RF

In [8]:
def build_equal_weight_benchmark(
    factor_returns: pd.DataFrame,
    tc: float = TC_FACTOR,
    warmup_months: int = 60
) -> dict:
    """
    Equal-weight static benchmark: w = [0.25, 0.25, 0.25, 0.25].
    TC is charged on rebalancing back to equal-weight after drift.
    """
    dates = factor_returns.index
    n_factors = factor_returns.shape[1]
    target_w = np.ones(n_factors) / n_factors

    net_returns = []
    turnover_series = []
    prev_w = target_w.copy()

    for t in range(warmup_months, len(dates) - 1):
        date_t1 = dates[t + 1]
        r_t1 = factor_returns.loc[date_t1].values

        # Rebalance to equal-weight each month
        turnover = np.abs(target_w - prev_w).sum()
        tc_cost = tc * turnover

        gross_ret = np.dot(target_w, r_t1)
        net_ret = gross_ret - tc_cost

        net_returns.append({'date': date_t1, 'return': net_ret})
        turnover_series.append({'date': date_t1, 'turnover': turnover})

        # Weights drift after return realisation
        drifted = target_w * (1 + r_t1)
        prev_w = drifted / drifted.sum()

    ret_s = pd.DataFrame(net_returns).set_index('date')['return']
    ret_s.name = 'Equal-Weight'
    nav = 100.0 * (1 + ret_s).cumprod()
    to_s = pd.DataFrame(turnover_series).set_index('date')['turnover']

    n_years = len(ret_s) / 12
    print(f'  Equal-Weight: {len(ret_s)} months, '
          f'Ann. Turnover={to_s.sum()/n_years:.1%}')

    return {'returns': ret_s, 'nav': nav, 'turnover': to_s}


def build_invvol_benchmark(
    factor_returns: pd.DataFrame,
    tc: float = TC_FACTOR,
    warmup_months: int = 60,
    vol_lookback: int = 36
) -> dict:
    """
    Inverse-volatility static benchmark: w_i = (1/sigma_i) / sum(1/sigma_j).
    Uses expanding window (with minimum vol_lookback months) for volatility.
    """
    dates = factor_returns.index
    n_factors = factor_returns.shape[1]

    net_returns = []
    turnover_series = []
    prev_w = np.ones(n_factors) / n_factors

    for t in range(warmup_months, len(dates) - 1):
        date_t = dates[t]
        date_t1 = dates[t + 1]

        # Expanding-window volatility (minimum vol_lookback months)
        start = max(0, t - vol_lookback + 1)
        vols = factor_returns.iloc[start:t+1].std()
        inv_vol = 1.0 / vols.values
        w_t = inv_vol / inv_vol.sum()

        turnover = np.abs(w_t - prev_w).sum()
        tc_cost = tc * turnover

        r_t1 = factor_returns.loc[date_t1].values
        gross_ret = np.dot(w_t, r_t1)
        net_ret = gross_ret - tc_cost

        net_returns.append({'date': date_t1, 'return': net_ret})
        turnover_series.append({'date': date_t1, 'turnover': turnover})
        prev_w = w_t.copy()

    ret_s = pd.DataFrame(net_returns).set_index('date')['return']
    ret_s.name = 'Inverse-Vol'
    nav = 100.0 * (1 + ret_s).cumprod()
    to_s = pd.DataFrame(turnover_series).set_index('date')['turnover']

    n_years = len(ret_s) / 12
    print(f'  Inverse-Vol: {len(ret_s)} months, '
          f'Ann. Turnover={to_s.sum()/n_years:.1%}')

    return {'returns': ret_s, 'nav': nav, 'turnover': to_s}


def build_market_benchmark(
    ff_data: pd.DataFrame,
    warmup_months: int = 60
) -> dict:
    """
    Market benchmark: Mkt-RF + RF = total market return.
    No transaction cost (passive index).
    """
    dates = ff_data.index
    mkt_total = ff_data['mkt_rf'] + ff_data['rf']

    live_ret = mkt_total.iloc[warmup_months + 1:].copy()
    live_ret.name = 'Market'
    nav = 100.0 * (1 + live_ret).cumprod()

    print(f'  Market: {len(live_ret)} months (no TC for passive index)')
    return {'returns': live_ret, 'nav': nav}


def build_6040_benchmark(
    ff_data: pd.DataFrame,
    warmup_months: int = 60
) -> dict:
    """
    60/40 benchmark: 60% market + 40% risk-free.
    r_6040 = 0.6 * (Mkt-RF + RF) + 0.4 * RF
    """
    dates = ff_data.index
    mkt_total = ff_data['mkt_rf'] + ff_data['rf']
    rf = ff_data['rf']

    ret_6040 = 0.6 * mkt_total + 0.4 * rf
    live_ret = ret_6040.iloc[warmup_months + 1:].copy()
    live_ret.name = '60/40'
    nav = 100.0 * (1 + live_ret).cumprod()

    print(f'  60/40: {len(live_ret)} months')
    return {'returns': live_ret, 'nav': nav}


print('Benchmark constructors defined.')

Benchmark constructors defined.


In [9]:
# ---- Build all benchmarks ----
print('Building benchmarks...')
print('=' * 60)

ew_result = build_equal_weight_benchmark(factor_ret, tc=TC_FACTOR, warmup_months=WARMUP_MONTHS)
iv_result = build_invvol_benchmark(factor_ret, tc=TC_FACTOR, warmup_months=WARMUP_MONTHS)
mkt_result = build_market_benchmark(ff_data, warmup_months=WARMUP_MONTHS)
s6040_result = build_6040_benchmark(ff_data, warmup_months=WARMUP_MONTHS)

print('=' * 60)
print('All benchmarks built.')

Building benchmarks...
  Equal-Weight: 64 months, Ann. Turnover=36.1%
  Inverse-Vol: 64 months, Ann. Turnover=31.9%
  Market: 64 months (no TC for passive index)
  60/40: 64 months
All benchmarks built.


In [10]:
# ---- Assemble all strategy and benchmark returns ----
all_strategies = {
    'BL Dynamic': bl_result,
    'CVaR Dynamic': cvar_result,
    'Equal-Weight': ew_result,
    'Inverse-Vol': iv_result,
    'Market': mkt_result,
    '60/40': s6040_result,
}

# Align all return series to common dates
returns_dict = {name: res['returns'] for name, res in all_strategies.items()}
returns_df = pd.DataFrame(returns_dict)
returns_df = returns_df.dropna(how='all')

# Build NAV series (base = 100)
nav_df = 100.0 * (1 + returns_df).cumprod()

print(f'\nBacktest period: {returns_df.index.min().strftime("%Y-%m")} to '
      f'{returns_df.index.max().strftime("%Y-%m")}')
print(f'Number of live months: {len(returns_df)}')
print(f'Strategies: {list(returns_df.columns)}')
print(f'\nFinal NAV values:')
print(nav_df.iloc[-1].round(2))


Backtest period: 2018-12 to 2024-03
Number of live months: 64
Strategies: ['BL Dynamic', 'CVaR Dynamic', 'Equal-Weight', 'Inverse-Vol', 'Market', '60/40']

Final NAV values:
BL Dynamic       85.24
CVaR Dynamic    109.17
Equal-Weight     84.83
Inverse-Vol     101.15
Market          203.59
60/40           163.54
Name: 2024-03-31 00:00:00, dtype: float64


---
## 9.4 Performance Metrics

Using `src/risk_metrics.py: compute_all_metrics()` for all strategies,
plus `benchmark_relative_metrics()` for benchmark-relative measures.

In [11]:
def compute_monthly_metrics(returns_series, rf_series=None, name='Strategy'):
    """
    Compute comprehensive metrics for monthly return series.
    Adapts compute_all_metrics (which uses 252-day annualisation)
    to work properly with monthly data.
    """
    r = returns_series.dropna()
    if len(r) == 0:
        return {'name': name}

    ann_ret = r.mean() * 12
    ann_vol = r.std() * np.sqrt(12)

    # Risk-free rate for Sharpe (annualised)
    if rf_series is not None:
        rf_ann = rf_series.reindex(r.index).fillna(0).mean() * 12
    else:
        rf_ann = 0.0

    excess_ret = ann_ret - rf_ann
    sharpe = excess_ret / ann_vol if ann_vol > 0 else 0

    # NAV and drawdown
    cum = (1 + r).cumprod()
    running_max = cum.cummax()
    drawdown = (cum - running_max) / running_max
    max_dd = drawdown.min()

    # Drawdown duration (in months)
    dd_duration = 0
    max_dd_duration = 0
    for i in range(len(drawdown)):
        if drawdown.iloc[i] < 0:
            dd_duration += 1
            max_dd_duration = max(max_dd_duration, dd_duration)
        else:
            dd_duration = 0

    # Sortino: downside deviation (monthly returns below zero)
    downside = r[r < 0]
    downside_dev = downside.std() * np.sqrt(12) if len(downside) > 0 else ann_vol
    sortino = excess_ret / downside_dev if downside_dev > 0 else 0

    # Calmar
    calmar = excess_ret / abs(max_dd) if max_dd != 0 else 0

    # VaR and CVaR (monthly)
    var_95 = -np.percentile(r, 5)
    cvar_95 = -r[r <= -var_95].mean() if (r <= -var_95).any() else var_95

    # Tail ratio
    right_tail = np.percentile(r, 95)
    left_tail = abs(np.percentile(r, 5))
    tail_ratio = right_tail / left_tail if left_tail > 0 else 0

    # Omega ratio
    gains = r[r > 0]
    losses = -r[r <= 0]
    omega = gains.sum() / losses.sum() if losses.sum() > 0 else np.inf

    return {
        'name': name,
        'ann_return': ann_ret,
        'ann_volatility': ann_vol,
        'sharpe': sharpe,
        'sortino': sortino,
        'max_drawdown': max_dd,
        'max_dd_duration_months': max_dd_duration,
        'calmar': calmar,
        'omega': omega,
        'var_95_monthly': var_95,
        'cvar_95_monthly': cvar_95,
        'skewness': r.skew(),
        'excess_kurtosis': r.kurtosis(),
        'hit_rate': (r > 0).mean(),
        'tail_ratio': tail_ratio,
        'n_months': len(r)
    }


# ---- Risk-free rate for Sharpe calculation ----
rf_live = ff_data['rf'].reindex(returns_df.index).fillna(0)

# ---- Compute metrics for all strategies ----
all_metrics = []
for col in returns_df.columns:
    m = compute_monthly_metrics(returns_df[col], rf_series=rf_live, name=col)
    # Add turnover info if available
    if col in all_strategies and 'turnover' in all_strategies[col]:
        to = all_strategies[col]['turnover']
        n_years = len(returns_df[col].dropna()) / 12
        m['ann_turnover'] = to.sum() / n_years if n_years > 0 else 0
        if 'tc_cost' in all_strategies[col]:
            tc_s = all_strategies[col]['tc_cost']
            m['ann_tc_bps'] = tc_s.sum() / n_years * 1e4 if n_years > 0 else 0
    all_metrics.append(m)

perf_df = pd.DataFrame(all_metrics).set_index('name')

# Display performance table
display_cols = [
    'ann_return', 'ann_volatility', 'sharpe', 'sortino',
    'max_drawdown', 'calmar', 'hit_rate', 'ann_turnover', 'ann_tc_bps'
]
display_cols = [c for c in display_cols if c in perf_df.columns]

print('\n=== PERFORMANCE SUMMARY ===')
print(perf_df[display_cols].round(4).to_string())


=== PERFORMANCE SUMMARY ===
              ann_return  ann_volatility  sharpe  sortino  max_drawdown  calmar  hit_rate  ann_turnover  ann_tc_bps
name                                                                                                               
BL Dynamic       -0.0248          0.1004 -0.4401  -0.5607       -0.2518 -0.1755    0.5156        0.0001      0.0018
CVaR Dynamic      0.0196          0.0800  0.0031   0.0053       -0.1800  0.0014    0.5625        0.1438      3.5951
Equal-Weight     -0.0258          0.1004 -0.4491  -0.5723       -0.2534 -0.1780    0.5156        0.3613         NaN
Inverse-Vol       0.0057          0.0844 -0.1619  -0.2246       -0.1768 -0.0773    0.5156        0.3186         NaN
Market            0.1524          0.1928  0.6901   1.0986       -0.2483  0.5359    0.6562           NaN         NaN
60/40             0.0992          0.1158  0.6898   1.0973       -0.1512  0.5280    0.6562           NaN         NaN


In [12]:
# ---- Benchmark-relative metrics (vs Equal-Weight) ----
print('\n=== BENCHMARK-RELATIVE METRICS (vs Equal-Weight) ===')

bench_ret = returns_df['Equal-Weight']
rel_metrics = []
for col in ['BL Dynamic', 'CVaR Dynamic', 'Inverse-Vol', 'Market', '60/40']:
    if col not in returns_df.columns:
        continue
    # Monthly series -- adapt to monthly frequency
    strat_ret = returns_df[col].dropna()
    common = strat_ret.index.intersection(bench_ret.dropna().index)
    if len(common) < 30:
        continue

    s = strat_ret.loc[common]
    b = bench_ret.loc[common]

    excess = s - b
    te = excess.std() * np.sqrt(12)
    ir = excess.mean() * 12 / te if te > 0 else 0

    # CAPM beta and alpha (monthly)
    cov_sb = np.cov(s, b)[0, 1]
    var_b = b.var()
    beta = cov_sb / var_b if var_b > 0 else 1.0
    alpha = (s.mean() - beta * b.mean()) * 12

    rel_metrics.append({
        'strategy': col,
        'information_ratio': ir,
        'tracking_error': te,
        'beta': beta,
        'alpha': alpha,
    })

rel_df = pd.DataFrame(rel_metrics).set_index('strategy')
print(rel_df.round(4).to_string())


=== BENCHMARK-RELATIVE METRICS (vs Equal-Weight) ===
              information_ratio  tracking_error    beta   alpha
strategy                                                       
BL Dynamic               5.7564          0.0002  0.9997  0.0009
CVaR Dynamic             0.6776          0.0669  0.5949  0.0349
Inverse-Vol              1.0127          0.0311  0.8051  0.0264
Market                   0.6822          0.2612 -1.0381  0.1257
60/40                    0.6577          0.1900 -0.6249  0.0831


---
## 9.5 Sharpe Significance: Jobson-Korkie Test

Test whether each dynamic strategy's Sharpe ratio is significantly different
from the equal-weight benchmark.

$$z_{JK} = \frac{SR_1 - SR_2}{\sqrt{\frac{1}{T}\left(2(1-\hat{\rho}) + \frac{1}{2}(SR_1^2 + SR_2^2 - 2 SR_1 SR_2 \hat{\rho}^2)\right)}}$$

In [13]:
def jobson_korkie_test(r1, r2, name1='Strategy', name2='Benchmark'):
    """
    Jobson-Korkie (1981) test for equality of two Sharpe ratios.
    Uses the Memmel (2003) correction for improved finite-sample properties.

    H0: SR1 = SR2
    H1: SR1 != SR2 (two-sided)

    Parameters
    ----------
    r1, r2 : pd.Series
        Return series (same frequency).

    Returns
    -------
    dict with z_stat, p_value, sr1, sr2, significant
    """
    # Align
    common = r1.dropna().index.intersection(r2.dropna().index)
    r1 = r1.loc[common].values
    r2 = r2.loc[common].values
    T = len(r1)

    mu1, mu2 = r1.mean(), r2.mean()
    s1, s2 = r1.std(ddof=1), r2.std(ddof=1)
    rho = np.corrcoef(r1, r2)[0, 1]

    sr1 = mu1 / s1 if s1 > 0 else 0
    sr2 = mu2 / s2 if s2 > 0 else 0

    # Variance of the difference (Memmel 2003 approximation)
    v = (1 / T) * (
        2 * (1 - rho)
        + 0.5 * (sr1**2 + sr2**2 - 2 * sr1 * sr2 * rho**2)
    )

    if v <= 0:
        return {
            'name1': name1, 'name2': name2,
            'sr1': sr1 * np.sqrt(12), 'sr2': sr2 * np.sqrt(12),
            'z_stat': np.nan, 'p_value': 1.0, 'significant_10pct': False
        }

    z_jk = (sr1 - sr2) / np.sqrt(v)
    p_value = 2 * (1 - norm.cdf(abs(z_jk)))  # Two-sided

    return {
        'name1': name1,
        'name2': name2,
        'sr1_ann': sr1 * np.sqrt(12),
        'sr2_ann': sr2 * np.sqrt(12),
        'sr_diff_ann': (sr1 - sr2) * np.sqrt(12),
        'z_stat': z_jk,
        'p_value': p_value,
        'significant_10pct': p_value < 0.10,
        'significant_5pct': p_value < 0.05,
        'T': T,
        'rho': rho,
    }


# ---- Test each dynamic strategy vs equal-weight ----
print('=== JOBSON-KORKIE SHARPE SIGNIFICANCE TESTS ===')
print('H0: SR_strategy = SR_equal_weight\n')

jk_results = []
ew_returns = returns_df['Equal-Weight']

for strat_name in ['BL Dynamic', 'CVaR Dynamic']:
    if strat_name not in returns_df.columns:
        continue
    jk = jobson_korkie_test(
        returns_df[strat_name], ew_returns,
        name1=strat_name, name2='Equal-Weight'
    )
    jk_results.append(jk)
    sig = 'YES' if jk['significant_10pct'] else 'NO'
    print(f"{strat_name} vs Equal-Weight:")
    print(f"  SR_strat={jk['sr1_ann']:.4f}, SR_ew={jk['sr2_ann']:.4f}, "
          f"diff={jk['sr_diff_ann']:.4f}")
    print(f"  z_JK={jk['z_stat']:.3f}, p={jk['p_value']:.4f}, "
          f"significant at 10%: {sig}")
    print(f"  (T={jk['T']} months, rho={jk['rho']:.3f})")
    print()

jk_df = pd.DataFrame(jk_results)
print(jk_df[['name1', 'sr1_ann', 'sr2_ann', 'sr_diff_ann', 'z_stat',
             'p_value', 'significant_10pct']].to_string(index=False))

=== JOBSON-KORKIE SHARPE SIGNIFICANCE TESTS ===
H0: SR_strategy = SR_equal_weight

BL Dynamic vs Equal-Weight:
  SR_strat=-0.2474, SR_ew=-0.2565, diff=0.0091
  z_JK=8.647, p=0.0000, significant at 10%: YES
  (T=64 months, rho=1.000)

CVaR Dynamic vs Equal-Weight:
  SR_strat=0.2451, SR_ew=-0.2565, diff=0.5016
  z_JK=1.616, p=0.1062, significant at 10%: NO
  (T=64 months, rho=0.747)

       name1   sr1_ann   sr2_ann  sr_diff_ann   z_stat  p_value  significant_10pct
  BL Dynamic -0.247419 -0.256477     0.009057 8.647255 0.000000               True
CVaR Dynamic  0.245107 -0.256477     0.501584 1.615699 0.106159              False


---
## 9.6 Sub-Period Analysis

Break the live period into economically meaningful sub-periods
and compute rolling 12-month Sharpe ratios.

In [14]:
# ---- Define sub-periods ----
SUB_PERIODS = {
    'Post-GFC Recovery (2010-2013)':  ('2010-01', '2013-12'),
    'QE Era (2014-2019)':             ('2014-01', '2019-12'),
    'COVID & Recovery (2020-2021)':   ('2020-01', '2021-12'),
    'Rate Shock & After (2022-2025)': ('2022-01', '2025-12'),
}

print('=== SUB-PERIOD ANALYSIS ===')
print()

sub_period_results = []

for period_name, (start, end) in SUB_PERIODS.items():
    mask = (returns_df.index >= start) & (returns_df.index <= end)
    sub_ret = returns_df.loc[mask]

    if len(sub_ret) < 6:
        print(f'{period_name}: insufficient data ({len(sub_ret)} months)')
        continue

    print(f'--- {period_name} ({len(sub_ret)} months) ---')

    for col in sub_ret.columns:
        r = sub_ret[col].dropna()
        if len(r) == 0:
            continue
        ann_ret = r.mean() * 12
        ann_vol = r.std() * np.sqrt(12)
        sharpe = ann_ret / ann_vol if ann_vol > 0 else 0
        cum = (1 + r).cumprod()
        max_dd = ((cum / cum.cummax()) - 1).min()

        sub_period_results.append({
            'period': period_name,
            'strategy': col,
            'ann_return': ann_ret,
            'ann_vol': ann_vol,
            'sharpe': sharpe,
            'max_dd': max_dd,
            'n_months': len(r),
        })

    # Quick summary per period
    sp_df = pd.DataFrame(
        [x for x in sub_period_results if x['period'] == period_name]
    ).set_index('strategy')[['ann_return', 'sharpe', 'max_dd']]
    print(sp_df.round(4).to_string())
    print()

sub_period_full = pd.DataFrame(sub_period_results)

=== SUB-PERIOD ANALYSIS ===

Post-GFC Recovery (2010-2013): insufficient data (0 months)
--- QE Era (2014-2019) (12 months) ---
              ann_return  sharpe  max_dd
strategy                                
BL Dynamic       -0.0482 -0.6068 -0.0577
CVaR Dynamic     -0.0444 -0.9627 -0.0406
Equal-Weight     -0.0489 -0.6160 -0.0579
Inverse-Vol      -0.0305 -0.5144 -0.0422
Market            0.1557  0.8600 -0.0671
60/40             0.1021  0.9402 -0.0394

--- COVID & Recovery (2020-2021) (23 months) ---
              ann_return  sharpe  max_dd
strategy                                
BL Dynamic       -0.0645 -0.6696 -0.2027
CVaR Dynamic     -0.0087 -0.1104 -0.1307
Equal-Weight     -0.0655 -0.6804 -0.2037
Inverse-Vol      -0.0111 -0.1413 -0.1379
Market            0.2301  1.1088 -0.2020
60/40             0.1390  1.1192 -0.1228

--- Rate Shock & After (2022-2025) (27 months) ---
              ann_return  sharpe  max_dd
strategy                                
BL Dynamic        0.0093  0.0815

In [15]:
# ---- Rolling 12-month Sharpe Ratio ----
ROLLING_WINDOW = 12  # months

rolling_sharpe = pd.DataFrame(index=returns_df.index)
for col in returns_df.columns:
    roll_mean = returns_df[col].rolling(ROLLING_WINDOW).mean() * 12
    roll_std = returns_df[col].rolling(ROLLING_WINDOW).std() * np.sqrt(12)
    rolling_sharpe[col] = roll_mean / roll_std

# Drop rows with insufficient data
rolling_sharpe = rolling_sharpe.dropna(how='all')

print(f'Rolling {ROLLING_WINDOW}-month Sharpe ratio computed.')
print(f'Date range: {rolling_sharpe.index.min()} to {rolling_sharpe.index.max()}')
print(f'\nCurrent (latest) rolling Sharpe:')
print(rolling_sharpe.iloc[-1].round(3))

Rolling 12-month Sharpe ratio computed.
Date range: 2019-11-30 00:00:00 to 2024-03-31 00:00:00

Current (latest) rolling Sharpe:
BL Dynamic     -0.499
CVaR Dynamic    0.557
Equal-Weight   -0.509
Inverse-Vol    -0.183
Market          1.899
60/40           2.136
Name: 2024-03-31 00:00:00, dtype: float64


---
## 9.7 Transaction Cost Sensitivity

Test backtest results at TC = 10, 25, 50 bps to assess robustness.

In [16]:
TC_LEVELS = [0.0010, 0.0025, 0.0050]  # 10, 25, 50 bps
TC_LABELS = ['10 bps', '25 bps', '50 bps']

print('=== TRANSACTION COST SENSITIVITY ===')
print()

tc_sensitivity = []

for tc_val, tc_label in zip(TC_LEVELS, TC_LABELS):
    for strat_name, weights_data in [('BL Dynamic', bl_weights), ('CVaR Dynamic', cvar_weights)]:
        result = run_walk_forward_backtest(
            factor_ret, weights_data, tc=tc_val,
            warmup_months=WARMUP_MONTHS,
            strategy_name=f'{strat_name} ({tc_label})'
        )
        r = result['returns'].dropna()
        ann_ret = r.mean() * 12
        ann_vol = r.std() * np.sqrt(12)
        sharpe = ann_ret / ann_vol if ann_vol > 0 else 0

        tc_sensitivity.append({
            'strategy': strat_name,
            'tc_bps': tc_val * 1e4,
            'ann_return': ann_ret,
            'ann_vol': ann_vol,
            'sharpe': sharpe,
            'ann_turnover': result['ann_turnover'],
            'ann_tc_drag_bps': result['ann_tc_drag'] * 1e4,
        })

tc_sens_df = pd.DataFrame(tc_sensitivity)
print('\n--- TC Sensitivity Summary ---')
print(tc_sens_df.round(4).to_string(index=False))

# Pivot for clarity
for strat in ['BL Dynamic', 'CVaR Dynamic']:
    sub = tc_sens_df[tc_sens_df['strategy'] == strat]
    print(f'\n{strat}:')
    print(sub[['tc_bps', 'ann_return', 'sharpe', 'ann_tc_drag_bps']].to_string(index=False))

=== TRANSACTION COST SENSITIVITY ===

  BL Dynamic (10 bps): 64 live months, Ann. Turnover=0.0%, Ann. TC drag=0.0 bps
  CVaR Dynamic (10 bps): 64 live months, Ann. Turnover=14.4%, Ann. TC drag=1.4 bps
  BL Dynamic (25 bps): 64 live months, Ann. Turnover=0.0%, Ann. TC drag=0.0 bps
  CVaR Dynamic (25 bps): 64 live months, Ann. Turnover=14.4%, Ann. TC drag=3.6 bps
  BL Dynamic (50 bps): 64 live months, Ann. Turnover=0.0%, Ann. TC drag=0.0 bps
  CVaR Dynamic (50 bps): 64 live months, Ann. Turnover=14.4%, Ann. TC drag=7.2 bps

--- TC Sensitivity Summary ---
    strategy  tc_bps  ann_return  ann_vol  sharpe  ann_turnover  ann_tc_drag_bps
  BL Dynamic    10.0     -0.0248   0.1004 -0.2474        0.0001           0.0007
CVaR Dynamic    10.0      0.0198   0.0800  0.2479        0.1438           1.4381
  BL Dynamic    25.0     -0.0248   0.1004 -0.2474        0.0001           0.0018
CVaR Dynamic    25.0      0.0196   0.0800  0.2451        0.1438           3.5951
  BL Dynamic    50.0     -0.0248   0

---
## 9.8 Visualizations

In [17]:
# ---- 9.8.1 NAV Curves (all strategies + benchmarks) ----
fig, ax = plt.subplots(figsize=(14, 7))

# Dynamic strategies: solid thick lines
dynamic_colors = {'BL Dynamic': COLORS['hml'], 'CVaR Dynamic': COLORS['rmw']}
bench_colors = {'Equal-Weight': '#7f8c8d', 'Inverse-Vol': '#95a5a6',
                'Market': '#2c3e50', '60/40': '#bdc3c7'}

for col in nav_df.columns:
    if col in dynamic_colors:
        ax.plot(nav_df.index, nav_df[col], label=col,
                color=dynamic_colors[col], linewidth=2.0)
    elif col in bench_colors:
        ax.plot(nav_df.index, nav_df[col], label=col,
                color=bench_colors[col], linewidth=1.2, linestyle='--', alpha=0.8)

ax.set_ylabel('NAV (base = 100)')
ax.set_title('Walk-Forward Backtest: NAV Curves', fontweight='bold', fontsize=14)
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_yscale('log')
ax.yaxis.set_major_formatter(mticker.ScalarFormatter())
ax.yaxis.set_minor_formatter(mticker.NullFormatter())
plt.tight_layout()

save_fig(fig, 'backtest_nav_curves')

Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/backtest_nav_curves.png


PosixPath('/Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/backtest_nav_curves.png')

In [18]:
# ---- 9.8.2 Drawdown Chart ----
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Top: Dynamic strategies
for col, color in dynamic_colors.items():
    if col not in nav_df.columns:
        continue
    dd = (nav_df[col] / nav_df[col].cummax()) - 1
    axes[0].fill_between(dd.index, dd, alpha=0.3, color=color)
    axes[0].plot(dd.index, dd, color=color, linewidth=1.0, label=col)

axes[0].set_ylabel('Drawdown')
axes[0].set_title('Dynamic Strategy Drawdowns', fontweight='bold')
axes[0].legend(loc='lower left')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
axes[0].grid(True, alpha=0.3)

# Bottom: All strategies comparison
all_colors = {**dynamic_colors, **bench_colors}
for col in nav_df.columns:
    color = all_colors.get(col, '#333333')
    dd = (nav_df[col] / nav_df[col].cummax()) - 1
    lw = 1.5 if col in dynamic_colors else 0.8
    ls = '-' if col in dynamic_colors else '--'
    axes[1].plot(dd.index, dd, color=color, linewidth=lw, linestyle=ls,
                label=col, alpha=0.9 if col in dynamic_colors else 0.6)

axes[1].set_ylabel('Drawdown')
axes[1].set_title('All Strategies: Drawdown Comparison', fontweight='bold')
axes[1].legend(loc='lower left', fontsize=8, ncol=3)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
save_fig(fig, 'backtest_drawdowns')

Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/backtest_drawdowns.png


PosixPath('/Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/backtest_drawdowns.png')

In [19]:
# ---- 9.8.3 Weight Evolution (BL and CVaR) ----
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

factor_colors = [COLORS.get(f, f'C{i}') for i, f in enumerate(FACTORS)]

# BL weights
if 'BL Dynamic' in all_strategies and 'weights' in all_strategies['BL Dynamic']:
    w_bl = all_strategies['BL Dynamic']['weights']
    axes[0].stackplot(
        w_bl.index, *[w_bl[f] for f in FACTORS],
        labels=FACTORS, colors=factor_colors, alpha=0.8
    )
    axes[0].set_ylabel('Weight')
    axes[0].set_ylim(0, 1)
    axes[0].set_title('Black-Litterman Dynamic Weights', fontweight='bold')
    axes[0].legend(loc='upper left', fontsize=9, ncol=4)
    axes[0].grid(True, alpha=0.3, axis='y')

# CVaR weights
if 'CVaR Dynamic' in all_strategies and 'weights' in all_strategies['CVaR Dynamic']:
    w_cv = all_strategies['CVaR Dynamic']['weights']
    axes[1].stackplot(
        w_cv.index, *[w_cv[f] for f in FACTORS],
        labels=FACTORS, colors=factor_colors, alpha=0.8
    )
    axes[1].set_ylabel('Weight')
    axes[1].set_ylim(0, 1)
    axes[1].set_title('CVaR Dynamic Weights', fontweight='bold')
    axes[1].legend(loc='upper left', fontsize=9, ncol=4)
    axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
save_fig(fig, 'backtest_weight_evolution')

Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/backtest_weight_evolution.png


PosixPath('/Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/backtest_weight_evolution.png')

In [20]:
# ---- 9.8.4 Rolling 12-Month Sharpe Ratio ----
fig, ax = plt.subplots(figsize=(14, 6))

for col in rolling_sharpe.columns:
    color = all_colors.get(col, '#333333')
    lw = 1.8 if col in dynamic_colors else 1.0
    ls = '-' if col in dynamic_colors else '--'
    alpha = 1.0 if col in dynamic_colors else 0.6
    ax.plot(rolling_sharpe.index, rolling_sharpe[col], color=color,
            linewidth=lw, linestyle=ls, label=col, alpha=alpha)

ax.axhline(y=0, color='black', linewidth=0.5)
ax.set_ylabel('Rolling 12-Month Sharpe Ratio')
ax.set_title('Rolling Sharpe Ratio (12-Month Window)', fontweight='bold', fontsize=14)
ax.legend(loc='upper left', fontsize=9, ncol=3)
ax.grid(True, alpha=0.3)
ax.set_ylim(-4, 6)
plt.tight_layout()

save_fig(fig, 'backtest_rolling_sharpe')

Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/backtest_rolling_sharpe.png


PosixPath('/Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/backtest_rolling_sharpe.png')

In [21]:
# ---- 9.8.5 Annual Return Bar Chart ----
annual_returns = returns_df.resample('YE').apply(lambda x: (1 + x).prod() - 1)

fig, ax = plt.subplots(figsize=(16, 7))

n_strategies = len(annual_returns.columns)
years = annual_returns.index.year
x = np.arange(len(years))
width = 0.8 / n_strategies

for i, col in enumerate(annual_returns.columns):
    color = all_colors.get(col, f'C{i}')
    ax.bar(x + i * width - 0.4 + width/2, annual_returns[col],
           width=width, label=col, color=color, alpha=0.85, edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(years, rotation=45)
ax.set_ylabel('Annual Return')
ax.set_title('Annual Returns by Strategy', fontweight='bold', fontsize=14)
ax.legend(loc='upper left', fontsize=8, ncol=3)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.axhline(y=0, color='black', linewidth=0.5)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()

save_fig(fig, 'backtest_annual_returns')

Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/backtest_annual_returns.png


PosixPath('/Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/backtest_annual_returns.png')

In [22]:
# ---- 9.8.6 Performance Summary Table (styled) ----
summary_display = perf_df[[
    'ann_return', 'ann_volatility', 'sharpe', 'sortino',
    'max_drawdown', 'calmar', 'hit_rate'
]].copy()

# Add turnover and TC where available
if 'ann_turnover' in perf_df.columns:
    summary_display['ann_turnover'] = perf_df['ann_turnover']
if 'ann_tc_bps' in perf_df.columns:
    summary_display['ann_tc_bps'] = perf_df['ann_tc_bps']

# Styled display
def highlight_best(s):
    """Highlight best value in each column."""
    if s.name in ['ann_return', 'sharpe', 'sortino', 'calmar', 'hit_rate']:
        best = s.max()
    elif s.name in ['ann_volatility', 'max_drawdown']:
        best = s.max()  # max drawdown closest to 0 is best
    elif s.name in ['ann_turnover', 'ann_tc_bps']:
        best = s.min()
    else:
        return ['' for _ in s]
    return ['font-weight: bold' if v == best else '' for v in s]

fmt = {
    'ann_return': '{:.2%}', 'ann_volatility': '{:.2%}',
    'sharpe': '{:.3f}', 'sortino': '{:.3f}',
    'max_drawdown': '{:.2%}', 'calmar': '{:.3f}',
    'hit_rate': '{:.1%}',
    'ann_turnover': '{:.1%}', 'ann_tc_bps': '{:.1f}',
}
fmt = {k: v for k, v in fmt.items() if k in summary_display.columns}

styled = summary_display.style.apply(highlight_best).format(fmt)
styled

,ann_return,ann_volatility,sharpe,sortino,max_drawdown,calmar,hit_rate,ann_turnover,ann_tc_bps
name,,,,,,,,,
BL Dynamic,-2.48%,10.04%,-0.440,-0.561,-25.18%,-0.176,51.6%,0.0%,0.0
CVaR Dynamic,1.96%,8.00%,0.003,0.005,-18.00%,0.001,56.2%,14.4%,3.6
Equal-Weight,-2.58%,10.04%,-0.449,-0.572,-25.34%,-0.178,51.6%,36.1%,nan
Inverse-Vol,0.57%,8.44%,-0.162,-0.225,-17.68%,-0.077,51.6%,31.9%,nan
Market,15.24%,19.28%,0.690,1.099,-24.83%,0.536,65.6%,nan%,nan
60/40,9.92%,11.58%,0.690,1.097,-15.12%,0.528,65.6%,nan%,nan


---
## 9.9 Validation Gates

In [23]:
print('=' * 60)
print('VALIDATION GATES --- Phase 9')
print('=' * 60)

gates_passed = 0
gates_total = 0
gate_results = []

# Helper
def check_gate(name, condition, detail=''):
    global gates_passed, gates_total
    gates_total += 1
    status = 'PASS' if condition else 'FAIL'
    if condition:
        gates_passed += 1
    symbol = '[PASS]' if condition else '[FAIL]'
    msg = f'{symbol} {name}'
    if detail:
        msg += f' -- {detail}'
    print(msg)
    gate_results.append({'gate': name, 'passed': condition, 'detail': detail})
    return condition


# Gate 1: BL outperforms equal-weight on Sharpe (>= 0.15 improvement)
if 'BL Dynamic' in perf_df.index and 'Equal-Weight' in perf_df.index:
    bl_sharpe = perf_df.loc['BL Dynamic', 'sharpe']
    ew_sharpe = perf_df.loc['Equal-Weight', 'sharpe']
    sharpe_diff_bl = bl_sharpe - ew_sharpe
    check_gate(
        'BL Sharpe > EW Sharpe + 0.15',
        sharpe_diff_bl >= 0.15,
        f'BL={bl_sharpe:.4f}, EW={ew_sharpe:.4f}, diff={sharpe_diff_bl:.4f}'
    )
else:
    check_gate('BL Sharpe > EW Sharpe + 0.15', False, 'Missing data')

# Gate 2: CVaR outperforms equal-weight on Sharpe (>= 0.15 improvement)
if 'CVaR Dynamic' in perf_df.index and 'Equal-Weight' in perf_df.index:
    cvar_sharpe = perf_df.loc['CVaR Dynamic', 'sharpe']
    sharpe_diff_cv = cvar_sharpe - ew_sharpe
    check_gate(
        'CVaR Sharpe > EW Sharpe + 0.15',
        sharpe_diff_cv >= 0.15,
        f'CVaR={cvar_sharpe:.4f}, EW={ew_sharpe:.4f}, diff={sharpe_diff_cv:.4f}'
    )
else:
    check_gate('CVaR Sharpe > EW Sharpe + 0.15', False, 'Missing data')

# Gate 3: Max drawdown of dynamic <= static
for dyn_name in ['BL Dynamic', 'CVaR Dynamic']:
    if dyn_name in perf_df.index and 'Equal-Weight' in perf_df.index:
        dyn_dd = perf_df.loc[dyn_name, 'max_drawdown']
        ew_dd = perf_df.loc['Equal-Weight', 'max_drawdown']
        check_gate(
            f'{dyn_name} max DD <= EW max DD',
            dyn_dd >= ew_dd,  # Both negative; closer to 0 is better
            f'{dyn_name}={dyn_dd:.2%}, EW={ew_dd:.2%}'
        )

# Gate 4: Turnover < 300% annually
for strat_name in ['BL Dynamic', 'CVaR Dynamic']:
    if strat_name in perf_df.index and 'ann_turnover' in perf_df.columns:
        to = perf_df.loc[strat_name, 'ann_turnover']
        if pd.notna(to):
            check_gate(
                f'{strat_name} turnover < 300%',
                to < 3.0,
                f'Ann. turnover = {to:.1%}'
            )

# Gate 5: TC < 100 bps annually
for strat_name in ['BL Dynamic', 'CVaR Dynamic']:
    if strat_name in perf_df.index and 'ann_tc_bps' in perf_df.columns:
        tc_bps = perf_df.loc[strat_name, 'ann_tc_bps']
        if pd.notna(tc_bps):
            check_gate(
                f'{strat_name} TC < 100 bps',
                tc_bps < 100,
                f'Ann. TC = {tc_bps:.1f} bps'
            )

# Gate 6: NAV strictly positive
for col in nav_df.columns:
    check_gate(
        f'{col} NAV > 0',
        (nav_df[col] > 0).all(),
        f'min NAV = {nav_df[col].min():.2f}'
    )

# Gate 7: Sharpe significance p < 0.10
for jk in jk_results:
    check_gate(
        f"{jk['name1']} JK p-value < 0.10",
        jk['p_value'] < 0.10 if not np.isnan(jk['p_value']) else False,
        f"p = {jk['p_value']:.4f}"
    )

print()
print(f'Gates passed: {gates_passed}/{gates_total}')

# Document failures honestly
failures = [g for g in gate_results if not g['passed']]
if failures:
    print(f'\nNOTE: {len(failures)} gate(s) did not pass. '
          'This is documented honestly as required by the protocol.')
    for f in failures:
        print(f"  - {f['gate']}: {f['detail']}")
    print('\nNegative results are a valid finding. Factor timing may not add '
          'sufficient value over static allocation after transaction costs '
          'in all regimes. See sub-period analysis for more nuance.')
else:
    print('\nAll validation gates passed.')

VALIDATION GATES --- Phase 9
[FAIL] BL Sharpe > EW Sharpe + 0.15 -- BL=-0.4401, EW=-0.4491, diff=0.0090
[PASS] CVaR Sharpe > EW Sharpe + 0.15 -- CVaR=0.0031, EW=-0.4491, diff=0.4523
[PASS] BL Dynamic max DD <= EW max DD -- BL Dynamic=-25.18%, EW=-25.34%
[PASS] CVaR Dynamic max DD <= EW max DD -- CVaR Dynamic=-18.00%, EW=-25.34%
[PASS] BL Dynamic turnover < 300% -- Ann. turnover = 0.0%
[PASS] CVaR Dynamic turnover < 300% -- Ann. turnover = 14.4%
[PASS] BL Dynamic TC < 100 bps -- Ann. TC = 0.0 bps
[PASS] CVaR Dynamic TC < 100 bps -- Ann. TC = 3.6 bps
[PASS] BL Dynamic NAV > 0 -- min NAV = 75.30
[PASS] CVaR Dynamic NAV > 0 -- min NAV = 81.59
[PASS] Equal-Weight NAV > 0 -- min NAV = 75.14
[PASS] Inverse-Vol NAV > 0 -- min NAV = 82.59
[PASS] Market NAV > 0 -- min NAV = 90.64
[PASS] 60/40 NAV > 0 -- min NAV = 94.46
[PASS] BL Dynamic JK p-value < 0.10 -- p = 0.0000
[FAIL] CVaR Dynamic JK p-value < 0.10 -- p = 0.1062

Gates passed: 14/16

NOTE: 2 gate(s) did not pass. This is documented honest

---
## 9.10 Save Outputs

In [24]:
# ---- Save backtest_returns.parquet ----
returns_df.to_parquet(BACKTEST_RETURNS_FILE)
print(f'Saved: {BACKTEST_RETURNS_FILE}')
print(f'  Shape: {returns_df.shape}')
print(f'  Columns: {list(returns_df.columns)}')
print(f'  Date range: {returns_df.index.min()} to {returns_df.index.max()}')
print(f'  NaN count: {returns_df.isna().sum().sum()}')
print()

# ---- Save backtest_nav.parquet ----
nav_df.to_parquet(BACKTEST_NAV_FILE)
print(f'Saved: {BACKTEST_NAV_FILE}')
print(f'  Shape: {nav_df.shape}')
print(f'  Final NAVs:')
print(nav_df.iloc[-1].round(2).to_string())
print()

# ---- Save performance summary to CSV ----
perf_df.to_csv(BACKTEST_PERF_FILE)
print(f'Saved: {BACKTEST_PERF_FILE}')
print(f'  Strategies: {list(perf_df.index)}')

Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/data/processed/backtest_returns.parquet
  Shape: (64, 6)
  Columns: ['BL Dynamic', 'CVaR Dynamic', 'Equal-Weight', 'Inverse-Vol', 'Market', '60/40']
  Date range: 2018-12-31 00:00:00 to 2024-03-31 00:00:00
  NaN count: 0

Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/data/processed/backtest_nav.parquet
  Shape: (64, 6)
  Final NAVs:
BL Dynamic       85.24
CVaR Dynamic    109.17
Equal-Weight     84.83
Inverse-Vol     101.15
Market          203.59
60/40           163.54

Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/tables/backtest_performance.csv
  Strategies: ['BL Dynamic', 'CVaR Dynamic', 'Equal-Weight', 'Inverse-Vol', 'Market', '60/40']


In [25]:
# ---- Final summary ----
print('\n' + '=' * 60)
print('PHASE 9 --- WALK-FORWARD BACKTEST ENGINE COMPLETE')
print('=' * 60)
print(f'Live period: {returns_df.index.min().strftime("%Y-%m")} to '
      f'{returns_df.index.max().strftime("%Y-%m")} '
      f'({len(returns_df)} months)')
print(f'Strategies tested: {len(returns_df.columns)}')
print(f'Validation gates: {gates_passed}/{gates_total} passed')
print()
print('Key files saved:')
print(f'  - {BACKTEST_RETURNS_FILE}')
print(f'  - {BACKTEST_NAV_FILE}')
print(f'  - {BACKTEST_PERF_FILE}')
print(f'  - Figures: backtest_nav_curves.png, backtest_drawdowns.png, etc.')

logger.info(f'Phase 9 complete. Gates: {gates_passed}/{gates_total}')
logger.info(f'Backtest period: {returns_df.index.min()} to {returns_df.index.max()}')


PHASE 9 --- WALK-FORWARD BACKTEST ENGINE COMPLETE
Live period: 2018-12 to 2024-03 (64 months)
Strategies tested: 6
Validation gates: 14/16 passed

Key files saved:
  - /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/data/processed/backtest_returns.parquet
  - /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/data/processed/backtest_nav.parquet
  - /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/tables/backtest_performance.csv
  - Figures: backtest_nav_curves.png, backtest_drawdowns.png, etc.
